<a href="https://colab.research.google.com/github/Azidian/Inteligencia_Artificial/blob/main/Componente_practico/Lecture_05/actividad_05_cheeseworld.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧀 Actividad 05 — Q-Learning a mano en CheeseWorld

En esta actividad se resuelven a mano (y se verifican con código) las actualizaciones de Q-Learning en **CheeseWorld (2 × 3)**.

La regla de actualización es:

$$Q(s_t,a_t)\leftarrow Q(s_t,a_t)+\alpha\left[r_{t+1}+\gamma\max_{a'}Q(s_{t+1},a')-Q(s_t,a_t)\right]$$

Los pasos son siempre los mismos:

1. TD target: $y = r + \gamma \max_{a'} Q(s',a')$
2. TD error: $\delta = y - Q(s,a)$
3. Actualización: $Q(s,a) \leftarrow Q(s,a) + \alpha\delta$

**Nota:** en estados terminales ($S_4$ y $S_5$) no hay acciones futuras, por lo tanto $\max_{a'}Q(s_{terminal},a')=0$.

In [ ]:
import numpy as np

ACCIONES = ["↑ Arriba", "→ Derecha", "↓ Abajo", "← Izquierda"]

def td_update(q_sa, r, max_q_next, alpha, gamma):
    """Devuelve (td_target, td_error, nuevo_q) para una transición."""
    td_target = r + gamma * max_q_next
    td_error = td_target - q_sa
    nuevo_q = q_sa + alpha * td_error
    return td_target, td_error, nuevo_q

## Ejercicio 1 — Una actualización de Q-Learning

El agente está en $S_0$ y ejecuta la acción → (Derecha). Se observa:

$$S_0 \xrightarrow{\rightarrow,\,+1} S_1$$

Q-table actual (orden de columnas: ↑, →, ↓, ←):

| Desde | ↑ | → | ↓ | ← |
|---|---|---|---|---|
| $S_0$ | 0.0 | 2.0 | 0.5 | 0.0 |
| $S_1$ | 1.0 | 4.0 | -2.0 | 0.5 |

Parámetros: $\alpha = 0.5$, $\gamma = 0.9$.

In [ ]:
Q_S0 = np.array([0.0, 2.0, 0.5, 0.0])
Q_S1 = np.array([1.0, 4.0, -2.0, 0.5])

alpha, gamma = 0.5, 0.9
r = 1

q_actual = Q_S0[1]          # Q(S0, →)
max_q_next = np.max(Q_S1)   # max Q(S1, a')

y, delta, q_nuevo = td_update(q_actual, r, max_q_next, alpha, gamma)

print("1. Q(S0, →) antes de actualizar:", q_actual)
print("2. max Q(S1, a') :", max_q_next)
print("3. TD target y   :", round(y, 2))
print("4. TD error δ    :", round(delta, 2))
print("5. Q(S0, →) nuevo:", round(q_nuevo, 2))

1. Q(S0, →) antes de actualizar: 2.0
2. max Q(S1, a') : 4.0
3. TD target y   : 4.6
4. TD error δ    : 2.6
5. Q(S0, →) nuevo: 3.3


**Respuestas:**

1. $Q(S_0,\rightarrow)$ antes de actualizar: **2.0**
2. $\max_{a'} Q(S_1,a')$: **4.0** (la mejor acción desde $S_1$ es →).
3. TD target: $y = 1 + 0.9(4.0) = 1 + 3.6 =$ **4.6**
4. TD error: $\delta = 4.6 - 2.0 =$ **2.6**
5. Actualización: $Q(S_0,\rightarrow) \leftarrow 2.0 + 0.5(2.6) = 2.0 + 1.3 =$ **3.3**

**Interpretación:** el TD error es positivo, es decir, el agente estaba subestimando el valor de ir a la derecha desde $S_0$. Al llegar a $S_1$ ve que desde allí todavía puede obtener un valor futuro alto (4.0), así que sube su estimación de 2.0 a 3.3.

## Ejercicio 2 — Recompensa negativa (estado terminal)

El agente está en $S_1$ y ejecuta la acción ↓ (Abajo). Se observa:

$$S_1 \xrightarrow{\downarrow,\,-10} S_4$$

donde $S_4$ es terminal (el veneno). Antes de actualizar: $Q(S_1,\downarrow)=1$. Parámetros: $\alpha = 0.5$, $\gamma = 0.9$.

In [ ]:
alpha, gamma = 0.5, 0.9
r = -10

q_actual = 1.0
max_q_next = 0.0   # S4 es terminal -> no hay acciones futuras

y, delta, q_nuevo = td_update(q_actual, r, max_q_next, alpha, gamma)

print("1. max Q(S4, a')  :", max_q_next)
print("2. TD target y    :", round(y, 2))
print("3. TD error δ     :", round(delta, 2))
print("4. Q(S1, ↓) nuevo :", round(q_nuevo, 2))

1. max Q(S4, a')  : 0.0
2. TD target y    : -10.0
3. TD error δ     : -11.0
4. Q(S1, ↓) nuevo : -4.5


**Respuestas:**

1. $\max_{a'} Q(S_4,a')=$ **0** (por ser estado terminal no hay acciones futuras).
2. TD target: $y = -10 + 0.9(0) =$ **-10**
3. TD error: $\delta = -10 - 1 =$ **-11**
4. Actualización: $Q(S_1,\downarrow) \leftarrow 1 + 0.5(-11) = 1 - 5.5 =$ **-4.5**

**Interpretación:** como $S_4$ es terminal, el target es simplemente la recompensa observada ($y=r$). El agente creía que ↓ valía +1, pero recibió -10; el TD error es muy negativo y la estimación baja a -4.5. Con más visitas, este valor seguirá acercándose a -10, y el agente aprenderá a evitar el veneno.

## Ejercicio 3 — Política ε-greedy

Desde $S_0$ la Q-table actual es:

| Desde | ↑ | → | ↓ | ← |
|---|---|---|---|---|
| $S_0$ | 0.2 | 3.3 | 1.1 | 0.0 |

Suponga $\epsilon = 0.1$.

In [ ]:
Q_S0 = np.array([0.2, 3.3, 1.1, 0.0])
epsilon = 0.1

mejor = int(np.argmax(Q_S0))
print("a. Acción al explotar:", ACCIONES[mejor])
print("b. Probabilidad de explorar:", epsilon)
print("c. Probabilidad de explotar:", round(1 - epsilon, 2))

a. Acción al explotar: → Derecha
b. Probabilidad de explorar: 0.1
c. Probabilidad de explotar: 0.9


**Respuestas:**

a. Si el agente explota, elige la acción con mayor valor Q: **→ (Derecha)**, con $Q=3.3$.

b. Probabilidad de explorar: $\epsilon =$ **0.1** (10 %).

c. Probabilidad de explotar: $1-\epsilon =$ **0.9** (90 %).

**Interpretación:** el agente casi siempre usa lo aprendido, pero el 10 % de las veces prueba una acción al azar. Esto evita que se quede atascado en una acción que *parece* buena solo porque no ha probado las demás.

## Ejercicio 4 — Dos pasos consecutivos

El agente realiza la secuencia:

$$S_0 \xrightarrow{\rightarrow,\,+1} S_1 \xrightarrow{\rightarrow,\,0} S_2$$

Q-table inicial: todos los valores en 0 para $S_0$, $S_1$ y $S_2$. Parámetros: $\alpha = 0.5$, $\gamma = 0.9$.

In [ ]:
alpha, gamma = 0.5, 0.9
# Orden de columnas: ↑, →, ↓, ←
Q = {
    "S0": np.zeros(4),
    "S1": np.zeros(4),
    "S2": np.zeros(4),
}
DER = 1

# --- 1) Transición S0 -> S1 con acción →, r = +1 ---
y1, d1, nuevo1 = td_update(Q["S0"][DER], 1, np.max(Q["S1"]), alpha, gamma)
Q["S0"][DER] = nuevo1
print("Paso 1: target =", round(y1, 2), "| error =", round(d1, 2), "| Q(S0,→) =", round(Q["S0"][DER], 2))

# --- 2) Transición S1 -> S2 con acción →, r = 0 (usa los Q-values ya actualizados) ---
y2, d2, nuevo2 = td_update(Q["S1"][DER], 0, np.max(Q["S2"]), alpha, gamma)
Q["S1"][DER] = nuevo2
print("Paso 2: target =", round(y2, 2), "| error =", round(d2, 2), "| Q(S1,→) =", round(Q["S1"][DER], 2))

Paso 1: target = 1.0 | error = 1.0 | Q(S0,→) = 0.5
Paso 2: target = 0.0 | error = 0.0 | Q(S1,→) = 0.0


**Respuestas:**

**1) Primera actualización, $S_0 \rightarrow S_1$:**

- $\max_{a'}Q(S_1,a') = 0$
- TD target: $y = 1 + 0.9(0) = 1$
- TD error: $\delta = 1 - 0 = 1$
- $Q(S_0,\rightarrow) \leftarrow 0 + 0.5(1) =$ **0.5**

**2) Segunda actualización, $S_1 \rightarrow S_2$:**

- $\max_{a'}Q(S_2,a') = 0$
- TD target: $y = 0 + 0.9(0) = 0$
- TD error: $\delta = 0 - 0 = 0$
- $Q(S_1,\rightarrow) \leftarrow 0 + 0.5(0) =$ **0**

**Interpretación:** en la primera actualización el agente aprende de la recompensa inmediata (+1) y $Q(S_0,\rightarrow)$ sube a 0.5. En la segunda, la recompensa es 0 y todos los valores de $S_2$ siguen en 0, así que no hay nada nuevo que aprender y $Q(S_1,\rightarrow)$ se queda en 0. Además, nótese que el valor aprendido en $S_0$ todavía no se ha propagado: la información del queso grande sólo llegará hacia atrás en episodios posteriores, cuando $S_2$ valga algo.

## Resumen de resultados

| Ejercicio | Resultado final |
|---|---|
| 1 | $Q(S_0,\rightarrow)=3.3$ (target 4.6, error 2.6) |
| 2 | $Q(S_1,\downarrow)=-4.5$ (target -10, error -11) |
| 3 | Explota → ; $P(\text{explorar})=0.1$ ; $P(\text{explotar})=0.9$ |
| 4 | $Q(S_0,\rightarrow)=0.5$ y luego $Q(S_1,\rightarrow)=0$ |